In [1]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=68bbb3cf2571e72460bad25b2759a78cd39e8b0faeb30b3dfeb3bfdf68e4567d
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [2]:
import os
import random
import warnings

from abc import ABC, abstractmethod
from dataclasses import dataclass

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from datasets import load_dataset
from huggingface_hub import login

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

from seqeval.metrics import (
    classification_report as seqeval_report,
    f1_score as seqeval_f1
)

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

warnings.filterwarnings("ignore")

In [3]:
@dataclass
class Config:

    # HF
    hf_token: str = ""

    # MODEL
    model_name: str = "ura-hcmut/ura-llama-2.1-8b"

    # INFERENCE
    batch_size: int = 8

    # GENERATION
    max_new_tokens: int = 32

    sample_ratio: float = 1
    # SEED
    seed: int = 42


cfg = Config()

In [4]:
def sample_dataframe(df, ratio=0.05, seed=42):

    return (
        df.sample(
            frac=ratio,
            random_state=seed
        )
        .reset_index(drop=True)
    )

In [5]:
class SeedManager:

    @staticmethod
    def set_seed(seed=42):

        random.seed(seed)
        np.random.seed(seed)

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


SeedManager.set_seed(cfg.seed)

In [6]:
login(cfg.hf_token)

In [7]:
class BaseDatasetProcessor(ABC):

    task_type = None

    def __init__(self, dataset_name):

        self.dataset_name = dataset_name
        self.dataset = None

    def load(self):

        self.dataset = load_dataset(
            self.dataset_name
        )

        return self

    @abstractmethod
    def get_test_dataframe(self):
        pass

In [8]:
class ViHSDProcessor(BaseDatasetProcessor):

    task_type = "classification"

    def __init__(self):

        super().__init__(
            "uitnlp/vihsd"
        )

        self.label_map = {
            0: "CLEAN",
            1: "OFFENSIVE",
            2: "HATE"
        }

    def get_test_dataframe(self):

        df = self.dataset["test"].to_pandas()

        df = df.rename(columns={
            "free_text": "text",
            "label_id": "label"
        })

        return df[["text", "label"]]

    def build_prompt(self, text: str):

        messages = [
            {
                "role": "system",
                "content": (
                    "Bạn là hệ thống phát hiện ngôn ngữ độc hại tiếng Việt.\n\n"
                    "Nhãn:\n"
                    "0 = CLEAN\n"
                    "1 = OFFENSIVE\n"
                    "2 = HATE\n\n"

                    "Định nghĩa:\n"
                    "- CLEAN: nội dung bình thường, không độc hại.\n"
                    "- OFFENSIVE: chửi tục, xúc phạm, công kích cá nhân, toxic nhưng không mang tính thù ghét nhóm/người.\n"
                    "- HATE: phát ngôn thù ghét, miệt thị, phân biệt vùng miền/tôn giáo/chủng tộc,"
                    " hoặc phản động/chống nhà nước/chính quyền.\n\n"
                    "- Kiểm tra theo thứ tự tính HATE, OFFENSIVE, CLEAN"

                    "Chỉ trả lời đúng duy nhất:\n"
                    "0 hoặc 1 hoặc 2"
                )
            },

            # FEW SHOT
            {
                "role": "user",
                "content": "bạn thật tuyệt vời"
            },
            {
                "role": "assistant",
                "content": "0"
            },

            {
                "role": "user",
                "content": "mày là đồ ngu"
            },
            {
                "role": "assistant",
                "content": "1"
            },

            {
                "role": "user",
                "content": "đồ súc vật mất dạy"
            },
            {
                "role": "assistant",
                "content": "1"
            },

            {
                "role": "user",
                "content": "hồ tặc hại dân hại nước"
            },
            {
                "role": "assistant",
                "content": "2"
            },

            {
                "role": "user",
                "content": "bọn mọi rợ nên cút khỏi đây"
            },
            {
                "role": "assistant",
                "content": "2"
            },

            {
                "role": "user",
                "content": "tôi yêu gia đình tôi"
            },
            {
                "role": "assistant",
                "content": "0"
            },

            # TARGET
            {
                "role": "user",
                "content": text
            }
        ]

        return messages

In [9]:
class ViCTSDProcessor(BaseDatasetProcessor):

    task_type = "classification"

    def __init__(self):

        super().__init__(
            "visolex/ViCTSD"
        )

        self.label_map = {
            0: "NON_TOXIC",
            1: "TOXIC"
        }

    def get_test_dataframe(self):

        df = (
            self.dataset["train"]
            .filter(lambda x: x["type"] == "test")
            .to_pandas()
        )

        df = df.rename(columns={
            "Comment": "text",
            "Toxicity": "label"
        })

        df = df[
            df["label"].isin([0, 1])
        ]

        return df[["text", "label"]]

    def build_prompt(self, text):
    
        return [
            {
                "role": "system",
                "content":
                    (
                        "Phân loại toxic comment tiếng Việt.\n\n"
    
                        "Nhãn:\n"
                        "0 = NON_TOXIC\n"
                        "1 = TOXIC\n\n"
    
                        "TOXIC bao gồm:\n"
                        "- chửi tục\n"
                        "- xúc phạm\n"
                        "- miệt thị\n"
                        "- công kích cá nhân\n"
                        "- sarcasm mang tính công kích\n"
                        "- toxic slang\n"
                        "- khinh thường\n"
                        "- toxic implicit theo ngữ cảnh\n"
                        "- lời nói mang tính hạ nhục hoặc ác ý\n\n"
    
                        "Một số toxic nhẹ hoặc cà khịa công kích "
                        "vẫn có thể là 1.\n\n"
    
                        "Không phải mọi câu tiêu cực đều là toxic.\n\n"
    
                        "Chỉ trả lời duy nhất:\n"
                        "0 hoặc 1"
                    )
            },
    
            # NON_TOXIC
            {
                "role": "user",
                "content": "Tôi thấy vụ này vẫn cần thêm thông tin"
            },
            {
                "role": "assistant",
                "content": "0"
            },
    
            {
                "role": "user",
                "content": "Không thích thì góp ý thôi chứ"
            },
            {
                "role": "assistant",
                "content": "0"
            },
    
            {
                "role": "user",
                "content": "Chắc do góc quay thôi"
            },
            {
                "role": "assistant",
                "content": "0"
            },
    
            {
                "role": "user",
                "content": "Hơi thất vọng thật"
            },
            {
                "role": "assistant",
                "content": "0"
            },
    
            # BORDERLINE TOXIC
            {
                "role": "user",
                "content": "Ns chuyện nghe đủ hiểu trình độ tới đâu rồi"
            },
            {
                "role": "assistant",
                "content": "1"
            },
    
            {
                "role": "user",
                "content": "Khôn như này quê tôi xích đầy"
            },
            {
                "role": "assistant",
                "content": "1"
            },
    
            # CLEAR TOXIC
            {
                "role": "user",
                "content": "Ngu thì im mẹ đi"
            },
            {
                "role": "assistant",
                "content": "1"
            },
    
            {
                "role": "user",
                "content": "Toàn một lũ phá hoại"
            },
            {
                "role": "assistant",
                "content": "1"
            },
    
            {
                "role": "user",
                "content": text
            }
        ]

In [10]:
class QwenModel:

    def __init__(self, model_name):

        self.model_name = model_name

        self.tokenizer = None
        self.model = None

    def load(self):

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name,
            trust_remote_code=True
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            device_map="auto",
            dtype=(
                torch.float16
                if torch.cuda.is_available()
                else torch.float32
            ),
            trust_remote_code=True
        )

        self.model.eval()

        return self

In [11]:
class GenerativeClassifier:

    def __init__(
        self,
        model,
        tokenizer,
        labels
    ):

        self.model = model
        self.tokenizer = tokenizer

        # ví dụ:
        # binary  -> ["0", "1"]
        # 3-class -> ["0", "1", "2"]
        self.labels = [
            str(label)
            for label in labels
        ]

    @torch.no_grad()
    def score_label(
        self,
        prompt_messages,
        label
    ):

        prompt_text = self.tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # thêm khoảng trắng để tokenization ổn định
        target_text = f"{label}"

        full_text = prompt_text + target_text

        full_enc = self.tokenizer(
            full_text,
            return_tensors="pt",
            add_special_tokens=False
        )

        prompt_enc = self.tokenizer(
            prompt_text,
            return_tensors="pt",
            add_special_tokens=False
        )

        input_ids = full_enc.input_ids.to(
            self.model.device
        )

        attention_mask = full_enc.attention_mask.to(
            self.model.device
        )

        labels_tensor = input_ids.clone()

        prompt_len = prompt_enc.input_ids.shape[1]

        # chỉ tính loss trên phần label
        labels_tensor[:, :prompt_len] = -100

        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_tensor
        )

        # negative loss = log-likelihood
        return -outputs.loss.item()

    @torch.no_grad()
    def predict(
        self,
        prompts
    ):

        all_preds = []
        all_probs = []

        for prompt in tqdm(prompts):

            scores = []

            for label in self.labels:

                score = self.score_label(
                    prompt_messages=prompt,
                    label=label
                )

                scores.append(score)

            scores = torch.tensor(
                scores,
                dtype=torch.float32
            )

            probs = F.softmax(
                scores,
                dim=-1
            )

            pred_idx = torch.argmax(
                probs
            ).item()

            # map về label thực
            pred_label = int(
                self.labels[pred_idx]
            )

            all_preds.append(pred_label)

            all_probs.append(
                probs.cpu().numpy()
            )

        return (
            np.array(all_preds),
            np.array(all_probs)
        )

In [12]:
class ClassificationEvaluator:

    @staticmethod
    def evaluate(
        y_true,
        y_pred,
        label_map,
        dataset_name
    ):

        print("\n")
        print("=" * 80)
        print(dataset_name)
        print("=" * 80)

        acc = accuracy_score(
            y_true,
            y_pred
        )

        mf1 = f1_score(
            y_true,
            y_pred,
            average="macro"
        )

        print(f"Accuracy : {acc:.4f}")
        print(f"MacroF1  : {mf1:.4f}")

        print("\nClassification Report:\n")

        print(
            classification_report(
                y_true,
                y_pred,
                target_names=list(
                    label_map.values()
                ),
                digits = 6,
            )
        )

In [13]:
vihsd = ViHSDProcessor().load()

victsd = ViCTSDProcessor().load()

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

dev.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/24048 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2672 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6680 [00:00<?, ? examples/s]

ViCTSD.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [14]:
model_wrapper = QwenModel(
    cfg.model_name
).load()

config.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [15]:
vihsd_df = vihsd.get_test_dataframe()
vihsd_df = sample_dataframe(
    vihsd_df,
    ratio=cfg.sample_ratio,
    seed=cfg.seed
)

vihsd_prompts = [
    vihsd.build_prompt(x)
    for x in vihsd_df["text"]
]

classifier = GenerativeClassifier(
    model=model_wrapper.model,
    tokenizer=model_wrapper.tokenizer,
    labels=["0", "1", "2"]
)

vihsd_preds, vihsd_probs = classifier.predict(
    vihsd_prompts
)

ClassificationEvaluator.evaluate(
    vihsd_df["label"],
    vihsd_preds,
    vihsd.label_map,
    "ViHSD"
)

vihsd_results = pd.DataFrame({
    "text": vihsd_df["text"],
    "label": vihsd_df["label"],
    "prediction": vihsd_preds,
    "prob_clean": vihsd_probs[:, 0],
    "prob_offensive": vihsd_probs[:, 1],
    "prob_hate": vihsd_probs[:, 2]
})

vihsd_correct_samples = vihsd_results[
    vihsd_results["label"] == vihsd_results["prediction"]
].head(20)

vihsd_wrong_samples = vihsd_results[
    vihsd_results["label"] != vihsd_results["prediction"]
].head(20)

print("CORRECT")
display(vihsd_correct_samples)

print("INCORRECT")
display(vihsd_wrong_samples)

  0%|          | 0/6680 [00:00<?, ?it/s]



ViHSD
Accuracy : 0.7575
MacroF1  : 0.4281

Classification Report:

              precision    recall  f1-score   support

       CLEAN   0.869518  0.864816  0.867161      5548
   OFFENSIVE   0.109208  0.114865  0.111965       444
        HATE   0.303597  0.306686  0.305134       688

    accuracy                       0.757485      6680
   macro avg   0.427441  0.428789  0.428086      6680
weighted avg   0.760696  0.757485  0.759080      6680

CORRECT


,text,label,prediction,prob_clean,prob_offensive,prob_hate
1,Đóm ơi lên là lênnnn,0,0,0.871492,0.049940,0.078567
2,"Sáng đánh răng bị chảy máu chân răng, để lâu k...",0,0,0.975853,0.020572,0.003575
3,Phong cách thả rông😂,0,0,0.706907,0.207335,0.085758
4,From Cà Mau with love (P9),0,0,0.943339,0.033045,0.023616
5,-_- rủi thiết bị giám sát hay DTDD nó quay lại...,0,0,0.958719,0.034921,0.006360
6,Choán thực sự :(( chia lên chia xuống kb bao n...,0,0,0.822552,0.066994,0.110454
7,Khi bạn vừa ngủ dậy chưa kịp kéo quần thì phải...,0,0,0.722254,0.230846,0.046899
8,e chồng của Hà Tăng Chung Gia Phú Trang Huỳnh,0,0,0.621061,0.148672,0.230267
9,Tuần sau lại được nghỉ rồi 🤭,0,0,0.918554,0.028396,0.053050
11,QuynhAnh DT thấy mà tứccccc,0,0,0.706076,0.152700,0.141224


INCORRECT


,text,label,prediction,prob_clean,prob_offensive,prob_hate
0,Inoue Kido photoshop,0,1,0.399254,0.441934,0.158812
10,Ghê vãi,1,2,0.064267,0.449600,0.486133
18,đm ông điêu ít thôiii :))),2,0,0.571651,0.265842,0.162506
28,Đm! Đen méo chơi nữa,0,2,0.046573,0.399197,0.554230
35,khok,0,1,0.055904,0.736388,0.207708
44,Với tài năng của thủ tướng cộng với trí tuệ củ...,2,0,0.876477,0.089538,0.033985
48,Người ta nói dân là sức mạnh của nước! Dần già...,0,2,0.079668,0.293698,0.626635
53,Vì thầy xạo lol nè,2,1,0.398499,0.417624,0.183877
54,1 cái cớ để rút ruột ăn bẩn thui,0,1,0.040791,0.628181,0.331028
55,Biết bao nhiêu vụ đi bắt người bị quay xong bả...,1,0,0.902621,0.086622,0.010758


In [16]:
vihsd_correct_samples = (
    vihsd_results[
        vihsd_results["label"] == vihsd_results["prediction"]
    ]
    .sort_values(
        by="label",
        ascending=False
    )
    .head(20)
)
display(vihsd_correct_samples)

,text,label,prediction,prob_clean,prob_offensive,prob_hate
4762,Túm lại: Muốn giết người mà vẫn vô tội hoặc án...,2,2,0.454506,0.087424,0.458070
2081,Cs toàn bọn nói láo,2,2,0.005871,0.156214,0.837915
1574,Dương =)) Làm gì kệ mẹ bạn,2,2,0.067291,0.174540,0.758168
1189,Đi làm osin mà xạo lol ghê,2,2,0.011540,0.140590,0.847869
2071,hệ thống thủy lợi yếu kém cộng thêm Khựa chơi ...,2,2,0.191673,0.118086,0.690241
5932,Con nay no đien roi,2,2,0.071306,0.367826,0.560868
3189,Cái lồn mẹ. Mấynhoom trước tao buk mình thì đỵ...,2,2,0.378021,0.158818,0.463161
5930,"K cần giặt đánh, cứ để dân tự chết vì ngu thôi :v",2,2,0.009454,0.070402,0.920144
3639,"Lý luận của cs với dân ..,khi cức chó...lộn kh...",2,2,0.416306,0.140539,0.443155
5208,co ai biet cach nao mo nhan xet e chui con Tra...,2,2,0.086375,0.094126,0.819500


In [17]:
victsd_df = victsd.get_test_dataframe()
victsd_df = sample_dataframe(
    victsd_df,
    ratio=cfg.sample_ratio,
    seed=cfg.seed
)

victsd_prompts = [
    victsd.build_prompt(x)
    for x in victsd_df["text"]
]

victsd_preds, victsd_probs = classifier.predict(
    victsd_prompts
)

ClassificationEvaluator.evaluate(
    victsd_df["label"],
    victsd_preds,
    victsd.label_map,
    "ViCTSD"
)

victsd_results = pd.DataFrame({
    "text": victsd_df["text"],
    "label": victsd_df["label"],
    "prediction": victsd_preds,
    "prob_non_toxic": victsd_probs[:, 0],
    "prob_toxic": victsd_probs[:, 1]
})

victsd_correct_samples = victsd_results[
    victsd_results["label"] == victsd_results["prediction"]
].head(20)

victsd_wrong_samples = victsd_results[
    victsd_results["label"] != victsd_results["prediction"]
].head(20)

print("CORRECT")
display(victsd_correct_samples)

print("INCORRECT")
display(victsd_wrong_samples)

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

  0%|          | 0/1000 [00:00<?, ?it/s]



ViCTSD
Accuracy : 0.3320
MacroF1  : 0.3149

Classification Report:

              precision    recall  f1-score   support

   NON_TOXIC   0.914179  0.275281  0.423143       890
       TOXIC   0.118852  0.790909  0.206651       110

    accuracy                       0.332000      1000
   macro avg   0.516516  0.533095  0.314897      1000
weighted avg   0.826693  0.332000  0.399329      1000

CORRECT


,text,label,prediction,prob_non_toxic,prob_toxic
0,Nên có 1 bộ luật linh động gặp những trường hợ...,0,0,0.664556,0.328979
12,Chó và rắn đều không phải là thức ăn! Nhắn nhủ...,1,1,0.298091,0.698519
15,"Tên Hùng này vừa côn đồ, vừa hèn hạ. Mong Cq C...",1,1,0.387930,0.605550
17,Thương quá là thương Bôm và a QT! Cố gắng lên ...,0,0,0.491842,0.488015
24,"Bất tài ,gian lận thương mại đã quá rõ ,đề ngh...",1,1,0.272322,0.723101
25,Cầm thú cũng không ác độc bằng hai chị em mụ này,1,1,0.357459,0.637239
26,Phải chăng chế tài xử phạt còn quá nhẹ. Haiz n...,1,1,0.400024,0.595834
27,Tôi sẽ không gây ra bất cứ sự xung đột nào với...,0,0,0.530088,0.464160
31,"Nếu theo thang điểm, tôi xin lỗi Ông:Nói tôi c...",0,0,0.523946,0.417726
34,Cá voi xanh thường bơi từ vùng biển bắc cực đế...,0,0,0.496376,0.496376


INCORRECT


,text,label,prediction,prob_non_toxic,prob_toxic
1,Hởi các Cẩu Tặc hảy đọc bài nầy mà bỏ nghề....,0,1,0.323517,0.674268
2,"Hổ dữ còn không ăn thịt con...Đồ khốn nạn, khô...",1,0,0.511533,0.484310
3,Giọng ca thế này thì nghệ sĩ trẻ còn phải luyệ...,0,1,0.363691,0.633330
4,"Nissan đang giảm giá kìa bạn, bạn quất luôn ch...",0,1,0.489950,0.505502
5,"Quá nguy hiểm, việc làm thẻ tín dụng hay làm g...",0,1,0.462046,0.531811
6,Bài viết hay và nhiều thông tin! Cảm ơn tác gi...,0,1,0.490623,0.506197
7,"chả hề gì, 2 tuần sau ông ấy sẽ vượt qua thôi",0,1,0.264688,0.730828
8,Dù con chó nào nhìn đẹp nhìn sạch đến đâu cũng...,0,1,0.373924,0.621332
9,"Phim đó tui xem khi đi thi đại học, tới giờ có...",0,1,0.261934,0.734611
10,"Trẻ em phải được vui chơi chạy nhảy, tại sao p...",1,0,0.540184,0.454880
